In [1]:
# ! pip install -q openai datasets pandas tqdm dotenv

### Imports

In [2]:
from datasets import load_dataset
from openai import OpenAI
import os
import json
import mlflow
import pandas as pd
from utils import generate_urls, calculate_invoice_accuracies, key_level_metrics, calculate_individual_invoice_accuracies
from prompt import register_prompt
from model import log_invoice_extraction_model

from dotenv import load_dotenv
from mlflow.entities import Feedback
from mlflow.genai import scorer

load_dotenv()

d:\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


True

### Config

In [3]:
MLFLOW_TRACKING_URI = "http://localhost:5000/"
MODEL_NAME = "gpt-5-nano"
REASONING = "low"
MLFLOW_EXPERIMENT_NAME = "cord-v2-gpt5-baseline"
PROMPT_NAME = "invoice-extraction-gpt5-prompt"
PROMPT_VERSION = "1"

### Initialize MLflow and OpenAI environment

In [4]:
client = OpenAI()
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME
# mlflow.openai.autolog()

### Register prompt and model

In [5]:
register_prompt(prompt_name=PROMPT_NAME)

You are a Vision Language Model designed to extract structured data from invoice receipts.
    Task:
    Convert the invoice receipt into a well-formed JSON object strictly following the schema provided.

    Requirements:
    1. Identify and extract only these sections (if present): `menu`, `sub_menu`, `sub_total`, `total`.  
    2. Preserve exact formatting for all the extracted values.  
    3. Do not output fields that lack data—omit empty keys.  
    4. Do not add any information not present in the invoice.
    5. In case of prices and currencies, ensure to maintain the original format without any modifications.

    Schema:
    {{schema}}

    Output:
    Return valid, minimal JSON matching this schema - no extraneous keys or null values.
    


2025/08/25 20:15:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: invoice-extraction-gpt5-prompt, version 6


### Load the dataset

In [8]:
dataset = load_dataset("naver-clova-ix/cord-v2")
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
    test: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
})

### Data Preparation

In [9]:
with open("schema.json", "r") as f:
    schema_dict = json.load(f)

schema_dict

{'menu': {'nm': 'name of the menu',
  'num': 'identification number of menu',
  'unitprice': 'unit price of menu',
  'cnt': 'quantity of menu',
  'discountprice': 'discounted price of menu',
  'price': 'total price of menu',
  'itemsubtotal': 'price of each menu after discount applied',
  'vatyn': 'whether the price includes tax or not',
  'etc': 'others',
  'sub': {'nm': 'name of submenu',
   'unitprice': 'unit price of submenu',
   'cnt': 'quantity of submenu',
   'price': 'total price of submenu',
   'etc': 'others'}},
 'sub_total': {'price': 'subtotal price',
  'discount_price': 'discounted price in total',
  'service_price': 'service charge',
  'othersvc_price': 'added charge other than service charge',
  'tax_price': 'tax amount',
  'etc': 'others'},
 'total': {'total_price': 'total price',
  'etc': 'others',
  'cashprice': 'amount of price paid in cash',
  'changeprice': 'amount of change in cash',
  'creditcardprice': 'amount of price paid in credit/debit card',
  'emoneyprice'

In [10]:
test_dataset = dataset["test"].select(range(3))
url_list, ground_truth_list = generate_urls(dataset=test_dataset)

100%|██████████| 3/3 [00:01<00:00,  1.66it/s]


### Inference and Evaluation

In [11]:
eval_dataset = []
for index, url in enumerate(url_list):
    eval_dict = {
        "inputs": {"image_base64": url, "schema": schema_dict},
        "expectations": {"expected_response" : ground_truth_list[index]}
    }
    eval_dataset.append(eval_dict)

eval_dataset

[{'inputs': {'image_base64': '...',
   'schema': {'menu': {'nm': 'name of the menu',
     'num': 'identification number of menu',
     'unitprice': 'unit price of menu',
     'cnt': 'quantity of menu',
     'discountprice': 'discounted price of menu',
     'price': 'total price of menu',
     'itemsubtotal': 'price of each menu after discount applied',
     'vatyn': 'whether the price includes tax or not',
     'etc': 'others',
     'sub': {'nm': 'name of submenu',
      'unitprice': 'unit price of submenu',
      'cnt': 'quantity of submenu',
      'price': 'total price of submenu',
      'etc': 'others'}},
    'sub_total': {'price': 'subtotal price',
     'discount_price': 'discounted price in total',
     'service_price': 'service charge',
     'othersvc_price': 'added charge other than service charge',
     'tax_price': 'tax amount',
     'etc': 'others'},
    'total': {'total_price': 'total price',
     'etc': 'others',
     'cashprice': 'amount of price paid in cash',
     'chang

In [12]:
# # @mlflow.trace(name="Evaluation")
# def perform_evaluation(outputs, expectations, trace_id):
#     pred_df, acc = calculate_individual_invoice_accuracies(ground_truth=expectations, output=outputs)
#     # mlflow.update_current_trace(tags={"ground_truth": json.dumps(expectations)})
#     mlflow.update_current_trace(tags={"ground_truth": "dummy_ground_truth"})
#     return pred_df, acc

In [13]:
@scorer
def exact_match(inputs, outputs, expectations, trace) -> Feedback:
    outputs = json.loads(outputs)
    expectations = expectations['expected_response']
    trace_id = trace.info.trace_id
    client_request_id = trace.info.client_request_id


    print("<<<<<<<<<Outputs: ", type(outputs))
    print("<<<<<<<<<Expectations: ", type(expectations))
    print("<<<<<<<<<<<<Trace ID: ", trace_id)
    print("<<<<<<<<<<<<Client Request ID: ", client_request_id)

    pred_df, acc = perform_evaluation(outputs=outputs, expectations=expectations, trace_id=trace_id)
    

    return Feedback(
        value=acc,
        name="Accuracy"
    )

In [14]:
@mlflow.trace
def predict_fn(image_base64, schema) -> str:
    system_prompt_template = mlflow.genai.load_prompt(name_or_uri=PROMPT_NAME, version=PROMPT_VERSION)
    system_prompt = system_prompt_template.format(schema=schema)

    response = client.responses.create(
        model=MODEL_NAME,
        reasoning={
            "effort": REASONING,
        },
        input=[
            {
                    "role": "user",
                    "content": [
                        {"type": "input_text", "text": system_prompt},
                        {"type": "input_image", "image_url": f"data:image/jpeg;base64,{image_base64}"}
                    ]
            }
        ],
    )
    response_text = response.output[1].content[0].text

    return response_text

In [15]:
if not os.path.exists("artifacts"):
    os.makedirs("artifacts")

In [16]:

with mlflow.start_run(run_name=f"{MODEL_NAME}-{REASONING}-evaluation") as run:

    @mlflow.trace(name="Evaluation")
    def perform_evaluation(outputs, expectations, trace_id):
        pred_df, acc = calculate_individual_invoice_accuracies(ground_truth=expectations, output=outputs)
        # mlflow.update_current_trace(tags={"ground_truth": json.dumps(expectations)})
        mlflow.update_current_trace(tags={"ground_truth": "dummy_ground_truth"})
        return pred_df, acc
    
    results = mlflow.genai.evaluate(
        data=eval_dataset,
        scorers=[
            exact_match
        ],
        predict_fn=predict_fn,
    )

2025/08/25 20:16:00 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2025/08/25 20:16:00 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset.


Evaluating:   0%|          | 0/3 [Elapsed: 00:00, Remaining: ?] 

<<<<<<<<<Outputs:  <class 'dict'>
<<<<<<<<<Expectations:  <class 'dict'>
<<<<<<<<<<<<Trace ID:  tr-4f4ce038e37b1a255437dc576da95191
<<<<<<<<<<<<Client Request ID:  None
<<<<<<<<<< GT Flat:  {'menu.nm': 'JASMINE MT ( L )', 'menu.cnt': '1', 'menu.price': '24.000', 'menu.sub.nm': 'COCONUT JELLY ( L )', 'menu.sub.price': '4.000', 'sub_total.subtotal_price': '28.000', 'total.total_price': '28.000', 'total.cashprice': '100.000', 'total.changeprice': '72.000', 'total.menuqty_cnt': '1'}
<<<<<<<<<< Pred Flat:  {'menu.nm': 'Coconut Jelly ( L )', 'menu.num': '', 'menu.unitprice': '4.000', 'menu.cnt': '1', 'menu.discountprice': '', 'menu.price': '4.000', 'menu.itemsubtotal': '4,000', 'menu.vatyn': '', 'menu.etc': '', 'menu.sub.nm': '', 'menu.sub.unitprice': '', 'menu.sub.cnt': '', 'menu.sub.price': '', 'menu.sub.etc': '', 'sub_total.price': '28.000', 'total.total_price': '28.000', 'total.etc': '', 'total.cashprice': '100.000', 'total.changeprice': '72.000', 'total.creditcardprice': '', 'total.emon